# Target24 — Генерация данных + Gold Trajectories (программные)

1. Генерация **train / val / test** для уровней 1–10 (без пересечений)
2. Генерация **gold trajectories** программно (AST-декомпозиция, без модели!)
3. Формирование **SFT dataset**

Hard levels: 8–10 (определены статически)

In [ ]:
# =========================================
# 0) Setup
# =========================================
UNZIP_HW2_ZIP = False
HW2_ZIP_PATH = "hw2.zip"
if UNZIP_HW2_ZIP:
    import os, zipfile
    with zipfile.ZipFile(HW2_ZIP_PATH) as z: z.extractall(".")
    print("Unzipped", HW2_ZIP_PATH)

# Раскомментировать при первом запуске в Colab:
# !pip -q install -U pip
# !pip -q install "transformers>=4.50" "trl>=0.22" datasets accelerate bitsandbytes peft
# !pip -q install unsloth huggingface_hub

In [ ]:
# =========================================
# 1) Imports
# =========================================
import os, sys, json, random, time, ast
from pathlib import Path
from collections import Counter

PROJECT_ROOT = os.environ.get("PROJECT_ROOT", ".")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from base.data import Data
from envs.target24.env import Target24Env, LEVEL_CONFIG, MAX_LEVEL

env = Target24Env()

for lvl, (nn, br, mn, mx, mt) in sorted(LEVEL_CONFIG.items()):
    print(f"  L{lvl:2d}: {nn} nums ({mn}-{mx}), brackets={'Y' if br else 'N'}")

In [ ]:
# =========================================
# 2) Config
# =========================================
DATA_DIR = Path("data_v2")
DATA_DIR.mkdir(exist_ok=True)

SAMPLES_PER_LEVEL = {
    1: (3000,500,500), 2: (3000,500,500), 3: (3000,500,500),
    4: (3000,500,500), 5: (3000,500,500), 6: (2000,400,400),
    7: (2000,400,400), 8: (1500,300,300), 9: (1500,300,300),
    10:(1000,200,200),
}
SEED = 42
HARD_LEVELS = [8, 9, 10]

# System prompt (единый для всех экспериментов)
SYSTEM_PROMPT = (
    "You are a helpful assistant. You always first think about the "
    "reasoning process in the mind and then provides the user with "
    "the answer.\n"
    "The reasoning process and answer are enclosed within "
    "'<think>' '</think>' and '<answer>' '</answer>' tags, "
    "respectively, e.g.,\n"
    "<think>\nA detailed reasoning process here.\n</think>\n"
    "<answer>\nReply to user here.\n</answer>. "
    "Please reason step by step, and put your final answer within "
    "<answer> </answer> tags."
)

print(f"Hard levels: {HARD_LEVELS}")

In [ ]:
# =========================================
# 3) Генерация: pool -> dedup -> split
# =========================================

def task_fp(d):
    m = d.metadata
    return (tuple(sorted(m['numbers'])), m['target'], m['gold_expr'])

def save_jsonl(data_list, path):
    with open(path, 'w', encoding='utf-8') as f:
        for d in data_list:
            f.write(json.dumps({'question': d.question, 'answer': d.answer,
                'difficulty': d.difficulty, 'metadata': d.metadata},
                ensure_ascii=False) + '\n')

all_data = {}

for level in range(1, MAX_LEVEL + 1):
    tr_n, va_n, te_n = SAMPLES_PER_LEVEL[level]
    total = tr_n + va_n + te_n

    t0 = time.time()
    pool = env.generate(num_of_questions=total*2, max_attempts=500,
                        difficulty=level, seed=SEED+level)
    dt = time.time() - t0

    seen, unique = set(), []
    for d in pool:
        fp = task_fp(d)
        if fp not in seen: seen.add(fp); unique.append(d)

    rng = random.Random(SEED + level*1000)
    rng.shuffle(unique)

    if len(unique) < total:
        r = len(unique) / total
        tr_n, va_n = int(tr_n*r), int(va_n*r)
        te_n = len(unique) - tr_n - va_n

    all_data[('train',level)] = unique[:tr_n]
    all_data[('val',  level)] = unique[tr_n:tr_n+va_n]
    all_data[('test', level)] = unique[tr_n+va_n:tr_n+va_n+te_n]

    for sp in ['train','val','test']:
        save_jsonl(all_data[(sp,level)], DATA_DIR / f'{sp}_L{level}.jsonl')

    print(f'L{level:2d}: {len(pool)}->{len(unique)} unique | '
          f'tr={len(all_data[("train",level)])} va={len(all_data[("val",level)])} '
          f'te={len(all_data[("test",level)])} | {dt:.1f}s')

# Проверка непересечения
print('\nOverlap check:')
for level in range(1, MAX_LEVEL+1):
    fps = {sp: {task_fp(d) for d in all_data[(sp,level)]} for sp in ['train','val','test']}
    ov = len(fps['train']&fps['val']) + len(fps['train']&fps['test']) + len(fps['val']&fps['test'])
    print(f'  L{level:2d}: {"OK" if ov==0 else f"OVERLAP={ov}"}')

In [ ]:
# =========================================
# 4) Combined + Hard файлы
# =========================================
for sp in ['train','val','test']:
    items = []
    for level in range(1, MAX_LEVEL+1):
        items.extend(all_data[(sp,level)])
    save_jsonl(items, DATA_DIR / f'{sp}.jsonl')
    print(f'{sp}.jsonl: {len(items)}')

for sp in ['train','val','test']:
    hard = []
    for l in HARD_LEVELS: hard.extend(all_data.get((sp,l), []))
    save_jsonl(hard, DATA_DIR / f'{sp}_hard.jsonl')
    print(f'{sp}_hard.jsonl: {len(hard)}')

## 2. Gold Trajectories (программная генерация)

Каждая задача содержит `gold_expr` — выражение, которое генератор использовал
для создания задачи. Мы парсим его в AST и раскладываем на пошаговые вычисления:

```
gold_expr: (12 + 3) * (7 - 5) + 11

→ <think>
  I need to find an expression using 12, 3, 7, 5, 11 that equals 41.
  Step 1: 12 + 3 = 15
  Step 2: 7 - 5 = 2
  Step 3: 15 * 2 = 30
  Step 4: 30 + 11 = 41
  The answer is (12 + 3) * (7 - 5) + 11.
  </think>
  <answer>
  (12 + 3) * (7 - 5) + 11
  </answer>
```

**Преимущества**: мгновенно, 100% корректно, не нужен GPU.

In [ ]:
# =========================================
# 5) Программная генерация gold chain-of-thought
# =========================================

# Маппинг AST-операторов в символы и функции
_OP_SYM = {ast.Add: '+', ast.Sub: '-', ast.Mult: '*', ast.Div: '/'}

def _eval_ast_node(node):
    """Вычисляет значение AST-узла (целочисленная арифметика)."""
    if isinstance(node, ast.Constant):
        return int(node.value)
    if isinstance(node, ast.BinOp):
        left = _eval_ast_node(node.left)
        right = _eval_ast_node(node.right)
        if left is None or right is None:
            return None
        op = type(node.op)
        if op == ast.Add:  return left + right
        if op == ast.Sub:  return left - right
        if op == ast.Mult: return left * right
        if op == ast.Div:
            if right == 0 or left % right != 0: return None
            return left // right
    return None

def _node_to_str(node):
    """AST-узел обратно в строку выражения."""
    if isinstance(node, ast.Constant):
        return str(node.value)
    if isinstance(node, ast.BinOp):
        left = _node_to_str(node.left)
        right = _node_to_str(node.right)
        op = _OP_SYM.get(type(node.op), '?')
        return f'({left} {op} {right})'
    return '?'

def _decompose_steps(node, steps):
    """
    Обходит AST post-order и записывает промежуточные шаги вычислений.
    Возвращает строковое представление текущего результата.
    """
    if isinstance(node, ast.Constant):
        return str(int(node.value))

    if isinstance(node, ast.BinOp):
        left_str = _decompose_steps(node.left, steps)
        right_str = _decompose_steps(node.right, steps)
        op_sym = _OP_SYM.get(type(node.op), '?')
        result = _eval_ast_node(node)
        if result is None:
            result = '?'

        step_num = len(steps) + 1
        steps.append(f'Step {step_num}: {left_str} {op_sym} {right_str} = {result}')
        return str(result)

    return '?'


def gold_expr_to_cot(gold_expr, numbers, target):
    """
    Конвертирует gold_expr в полный chain-of-thought ответ:
    <think>рассуждение</think><answer>выражение</answer>

    Args:
        gold_expr: строка выражения, например '(12 + 3) * (7 - 5) + 11'
        numbers: список чисел, например ['12', '3', '7', '5', '11']
        target: целевое число, например '41'

    Returns:
        Полный текст ответа с CoT
    """
    try:
        tree = ast.parse(gold_expr, mode='eval')
    except SyntaxError:
        # Fallback если выражение не парсится
        return (f'<think>\nI need to combine {", ".join(numbers)} to get {target}.\n'
                f'The answer is {gold_expr}.\n</think>\n'
                f'<answer>\n{gold_expr}\n</answer>')

    steps = []
    _decompose_steps(tree.body, steps)

    nums_str = ', '.join(numbers)
    reasoning_lines = [
        f'I need to combine {nums_str} to get {target}.',
        f'Let me compute {gold_expr} step by step:',
    ]
    reasoning_lines.extend(steps)
    reasoning_lines.append(f'The result is {target}, which matches the target.')

    reasoning = '\n'.join(reasoning_lines)
    return (f'<think>\n{reasoning}\n</think>\n'
            f'<answer>\n{gold_expr}\n</answer>')


# === Тест ===
test_expr = '(5 + 3) * 2'
test_cot = gold_expr_to_cot(test_expr, ['5', '3', '2'], '16')
print('Example gold CoT:')
print(test_cot)
print()

# Тест на сложном выражении
test2 = '((12 + 3) * (7 - 5)) + 11'
print(gold_expr_to_cot(test2, ['12','3','7','5','11'], '41'))

In [ ]:
# =========================================
# 6) Генерация gold CoT для всех уровней
# =========================================

GOLD_DIR = DATA_DIR / 'gold'
GOLD_DIR.mkdir(exist_ok=True)

gold_results = {}

for level in range(1, MAX_LEVEL + 1):
    train_data = all_data.get(('train', level), [])
    results = []
    ok, fail = 0, 0

    for d in train_data:
        m = d.metadata
        gold_expr = m['gold_expr']
        numbers = m['numbers']
        target = m['target']

        # Генерируем chain-of-thought программно
        cot = gold_expr_to_cot(gold_expr, numbers, target)

        # Верифицируем что ответ корректный
        verified = env.verify(d, cot)

        results.append({
            'question': d.question,
            'answer': d.answer,
            'difficulty': d.difficulty,
            'metadata': m,
            'gold_completion': cot,
            'gold_verified': verified,
        })
        if verified: ok += 1
        else: fail += 1

    gold_results[level] = results

    # Сохраняем
    save_path = GOLD_DIR / f'gold_train_L{level}.jsonl'
    with open(save_path, 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

    print(f'L{level:2d}: {ok}/{len(results)} verified '
          f'({ok/max(1,len(results))*100:.0f}%) -> {save_path}')

print('\nAll gold generated!')

In [ ]:
# =========================================
# 7) Формируем SFT dataset
# =========================================

sft_data = []
print('Gold summary:')
for level in range(1, MAX_LEVEL + 1):
    res = gold_results.get(level, [])
    ok = sum(1 for r in res if r['gold_verified'])
    print(f'  L{level:2d}: {ok}/{len(res)}')
    for r in res:
        if not r['gold_verified']: continue
        sft_data.append({
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': r['question']},
                {'role': 'assistant', 'content': r['gold_completion']},
            ],
            'difficulty': r['difficulty'],
            'metadata': r['metadata'],
        })

sft_path = DATA_DIR / 'sft_gold.jsonl'
with open(sft_path, 'w', encoding='utf-8') as f:
    for row in sft_data:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'\nSFT dataset: {len(sft_data)} examples -> {sft_path}')
lc = Counter(r['difficulty'] for r in sft_data)
for l in sorted(lc): print(f'  L{l}: {lc[l]}')

In [ ]:
# =========================================
# 8) Примеры gold CoT по уровням
# =========================================

for level in [1, 3, 5, 7, 10]:
    res = gold_results.get(level, [])
    ok_list = [r for r in res if r['gold_verified']]
    if ok_list:
        r = ok_list[0]
        print(f'\n=== Level {level} ===')
        print(f'Numbers: {r["metadata"]["numbers"]}, Target: {r["metadata"]["target"]}')
        print(r['gold_completion'])
        print()

In [ ]:
# =========================================
# 9) Скачать результаты (Colab)
# =========================================

# Итоговая статистика
print('\nFiles in data_v2/:')
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        lines = sum(1 for _ in open(f, encoding='utf-8'))
        print(f'  {f.name}: {lines} lines')
if (DATA_DIR / 'gold').exists():
    print('  gold/:')
    for f in sorted((DATA_DIR / 'gold').iterdir()):
        lines = sum(1 for _ in open(f, encoding='utf-8'))
        print(f'    {f.name}: {lines} lines')

print('\nReady for GRPO_full_experiment.ipynb!')

# Раскомментировать чтобы скачать:
# import shutil
# shutil.make_archive('data_v2', 'zip', '.', 'data_v2')
# from google.colab import files
# files.download('data_v2.zip')